In [55]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [54]:
gilt_eps = 6e-6
chi = 30
trunc_shape = [14 14; 14 14; 14 14; 14 14]  # shape to truncate to, not to deal with Gilt tensor dimension oscillations
cg_eps = 1e-10
newton_eps = 1e-9
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 1,
	"rotate" => true,
    "bond_repetitions" =>2,
    "recursion_depth" => Dict(
		"S" => 60,
		"N" => 60,
		"E" => 60,
		"W" => 60,
	)
)
Jratio = 1.0

relT=1.0
rg_steps = 10
#do rg_steps steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, rg_steps, gilt_pars)["A"];
#NB traj consists of PyObjects

traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[rg_steps+1], accepted_elements, _ = fix_discrete_gauge(traj[rg_steps+1]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

for i in 1:length(traj)
    println(i," ",traj[i].shape, traj[i].qhape )
end

/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return sinh(2*x*Jv)*sinh(2*x*Jh) - 1
/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return sinh(2*x*Jv)*sinh(2*x*Jh) - 1
/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array 

1 

┌ Warning: new_list_of_elements: new entry is below the threshold. It was -2.0866645186969456e-5 and became 0.0. Index CartesianIndex(8, 15, 3, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 0.000223097448272664 and became -5.739677069936619e-8. Index CartesianIndex(1, 18, 24, 3) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was 2.2052614892838678e-5 and became 0.0. Index CartesianIndex(1, 2, 15, 5) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It was -2.0866645186969456e-5 and became 0.0. Index CartesianIndex(8, 15, 3, 1) 
└ @ Main ~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GaugeFixing.jl:466
┌ Warning: new_list_of_elements: new entry is below the threshold. It wa

[1 1; 1 1; 1 1; 1 1][0 1; 0 1; 0 1; 0 1]
2 [2 2; 2 2; 2 2; 2 2][0 1; 0 1; 0 1; 0 1]
3 [8 8; 8 8; 8 8; 8 8][0 1; 0 1; 0 1; 0 1]
4 [14 16; 14 16; 14 16; 14 16][0 1; 0 1; 0 1; 0 1]
5 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
6 [15 15; 14 16; 15 15; 14 16][0 1; 0 1; 0 1; 0 1]
7 [14 16; 14 16; 14 16; 14 16][0 1; 0 1; 0 1; 0 1]
8 [14 16; 14 16; 14 16; 14 16][0 1; 0 1; 0 1; 0 1]
9 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
10 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]
11 [15 15; 15 15; 15 15; 15 15][0 1; 0 1; 0 1; 0 1]


In [ ]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

A[1] = truncate_blocks(traj[7], trunc_shape)

In [61]:
for i in 2:30
    println("i=",i)

    RA = gilt_with_cont_gauge(A[i], gilt_pars; trunc_shape = trunc_shape);
    RA, accepted_elements[i] = fix_discrete_gauge(RA; tol = 1e-7);
    
    A[i], _ = fix_discrete_gauge(ju_to_py(A[i]), accepted_elements[i])
    A[i] = py_to_ju(A[i])

    e0 = embedded_distance(RA, A[i])
    println("||R(A[i])-A[i]||= ", e0)
    println("shapes:", A[i].shape, RA.shape)
    flush(stdout)

    gradient = gradient_descent_with_iter_fixed(A[i], accepted_elements[i], gilt_pars; 
        trunc_shape = trunc_shape);

    println(norm(gradient))

    if i==1
        A[i+1] = A[i] - 0.2*gradient
        continue
    end

    function phi(a)
        A0 = A[i]- a * gradient
        
        A0, _ = fix_discrete_gauge(ju_to_py(A0), accepted_elements[i])
        A0 = py_to_ju(A0)
        
        RA = gilt_with_cont_gauge(A0, gilt_pars; trunc_shape = trunc_shape);
        RA, _ = fix_discrete_gauge(ju_to_py(RA), accepted_elements[i]);
        RA = py_to_ju(RA)
    
        return embedded_distance(RA, A0)
    end

    
    println(phi.(0.001:0.001:0.01))
    #println(phi.(0.31:0.01:0.39))
        
    #A[i+1]=A[i]-0.1*gradient
    
    throw()
    
    deltaA[i] = newton_correction_with_iterations_fixed(A[i], 10, accepted_elements[i], gilt_pars; 
        trunc_shape = trunc_shape);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    newton_step = 0.25
    enew = e0
    while true #damped Newton method implementation, which reduces a step by 2 if cost function does not decrease
        println("newton_step= ", newton_step)
        Anew = A[i] + newton_step * deltaA[i]

        RAnew = gilt_with_cont_gauge(Anew, gilt_pars; trunc_shape = trunc_shape);
        RAnew, accepted_elements_new = fix_discrete_gauge(RAnew; tol = 1e-7);
    
        Anew, _ = fix_discrete_gauge(ju_to_py(Anew), accepted_elements_new)
        Anew = py_to_ju(Anew)

        enew = embedded_distance(RAnew, Anew)
        
        println("fp_error= ", enew)
        println("shapes:", Anew.shape, RAnew.shape)
        if enew < e0 && Anew.shape == RAnew.shape
            A[i+1] = Anew
            break
        end
        newton_step *= 0.5 
    end
    if enew < newton_eps
        break
    end
end

i=2
||R(A[i])-A[i]||= 0.06962356596152251
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
0.06930102658478698
[0.06962895283002489, 0.06963440417434852, 0.06963991753401107, 0.06964549458773577, 0.06965114131249915, 0.06965684817810497, 0.06966261749831586, 0.06966844521410355, 0.06967432920591111, 0.06968026773799302]


LoadError: ArgumentError: throw: too few arguments (expected 1)